In [49]:
import sqlite3
import pandas as pd

import os
from openai import OpenAI
import numpy as np
import pickle
import time

import fitz  # PyMuPDF
import re

import requests



In [64]:
# Connect to the provided SQLite database
db_path = 'data/financials.db'
conn = sqlite3.connect(db_path)

# List all tables in the database
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in Database:\n", tables, "\n")

for table in tables['name']:
    columns = pd.read_sql_query("PRAGMA table_info({table_name});".format(table_name=table), conn)
    print("\nTable: {table_name}:\n".format(table_name=table), columns[['name', 'type']])
    cols = ', '.join(list(columns['name']))
    print(table + ': (' + cols + ')')
    

# Get column names for the Income Statement
# columns = pd.read_sql_query("PRAGMA table_info(income_statements);", conn)
# print("Income Statement Schema:\n", columns[['name', 'type']])

conn.close()

Tables in Database:
                  name
0           companies
1   income_statements
2     sqlite_sequence
3      balance_sheets
4     segment_revenue
5  geographic_revenue 


Table: companies:
               name  type
0           ticker  TEXT
1             name  TEXT
2              cik  TEXT
3              sic  TEXT
4           sector  TEXT
5  fiscal_year_end  TEXT
companies: (ticker, name, cik, sic, sector, fiscal_year_end)

Table: income_statements:
                         name     type
0                         id  INTEGER
1             company_ticker     TEXT
2                fiscal_year  INTEGER
3               period_start     TEXT
4                 period_end     TEXT
5                period_type     TEXT
6                    revenue   BIGINT
7            cost_of_revenue   BIGINT
8               gross_profit   BIGINT
9   research_and_development   BIGINT
10  total_operating_expenses   BIGINT
11          operating_income   BIGINT
12                net_income   BIGINT
13     

In [38]:
PDF_FILES = [
    "data/pdfs/AAPL_FY2024_10-K.pdf", "data/pdfs/AAPL_FY2025_10-K.pdf",
    "data/pdfs/GOOGL_FY2024_10-K.pdf", "data/pdfs/GOOGL_FY2025_10-K.pdf",
    "data/pdfs/MSFT_FY2024_10-K.pdf", "data/pdfs/MSFT_FY2025_10-K.pdf"
]
!ls data/pdfs

AAPL_FY2024_10-K.pdf  GOOGL_FY2024_10-K.pdf MSFT_FY2024_10-K.pdf
AAPL_FY2025_10-K.pdf  GOOGL_FY2025_10-K.pdf MSFT_FY2025_10-K.pdf


In [29]:
def get_pdf_chunks(pdf_path, chunk_size=1000, overlap=200):
    doc = fitz.open(pdf_path)
    full_text = ""
    
    # 1. Extraction
    for page in doc:
        # "text" mode preserves natural reading order
        full_text += page.get_text("text") + "\n"
    
    # 2. Cleaning (Optional but recommended)
    # Remove excessive whitespace/newlines that often occur in PDF tables
    full_text = re.sub(r'\n+', '\n', full_text)
    
    # 3. Chunking with Overlap
    chunks = []
    start = 0
    while start < len(full_text):
        end = start + chunk_size
        chunk = full_text[start:end]
        chunks.append(chunk)
        start += (chunk_size - overlap)
    
    return chunks

# Test it on the Apple 10-K
chunks = get_pdf_chunks("data/pdfs/AAPL_FY2024_10-K.pdf")
print(f"Total chunks created: {len(chunks)}")
print("-" * 30)
print(f"Sample from Chunk 5:\n{chunks[5][:500]}...")

Total chunks created: 257
------------------------------
Sample from Chunk 5:
. ☐
Indicate by check mark whether the Registrant is a shell company (as defined in Rule 12b-2 of the Act).
Yes  ☐     No  ☒
The aggregate market value of the voting and non-voting stock held by non-affiliates of the Registrant, as of March 29, 2024, the last business day of
the Registrant’s most recently completed second fiscal quarter, was approximately $2,628,553,000,000. Solely for purposes of this disclosure, shares
of common stock held by executive officers and directors of the Registrant ...


In [47]:
os.environ["FIREWORKS_API_KEY"] = "fw_SP1n5hfwEKiUvTQ3jH3WnE"
client = OpenAI(
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=os.getenv("FIREWORKS_API_KEY"),
)

EMBED_MODEL = "nomic-ai/nomic-embed-text-v1.5"

def get_embeddings(texts, batch_size=100):
    """Fetches embeddings in batches to avoid API row limits."""
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        print(f"Processing batch {i//batch_size + 1} ({len(batch)} chunks)...")
        
        try:
            response = client.embeddings.create(
                input=batch,
                model=EMBED_MODEL
            )
            all_embeddings.extend([e.embedding for e in response.data])
        except Exception as e:
            print(f"Error in batch starting at index {i}: {e}")
            # Optional: add a small sleep if you hit rate limits
            # time.sleep(1) 
            
    return np.array(all_embeddings)

# 1. Vectorize your chunks in batches
print(f"Vectorizing {len(chunks)} chunks total...")
chunk_embeddings = get_embeddings(chunks, batch_size=100)

def vector_search(query, chunk_embeddings, chunks, top_k=3):
    """Performs vector search over 10-K chunks"""
    # You need your actual vector search logic here
    # 1. Get embedding for the user's tool-call query
    resp = client.embeddings.create(
        input=[query],
        model="nomic-ai/nomic-embed-text-v1.5"
    )
    query_emb = np.array(resp.data[0].embedding)
    
    # 2. Simple cosine similarity using the global variables
    similarities = np.dot(chunk_embeddings, query_emb) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    
    # 3. Get top 5 results
    top_indices = np.argsort(similarities)[-5:][::-1]
    results = [chunks[i] for i in top_indices]
    
    return "\n---\n".join(results)

# Test Search again
print("\nTesting search...")
results = vector_search("What are the main risk factors for Apple's supply chain?", chunk_embeddings, chunks)
if results:
    print("\nTop Search Result:")
    print(results[0][:500] + "...")

Vectorizing 257 chunks total...
Processing batch 1 (100 chunks)...
Processing batch 2 (100 chunks)...
Processing batch 3 (57 chunks)...

Testing search...

Top Search Result:
t...


In [41]:
def ingest_all():
    all_chunks = []
    # Loop through each PDF, extract text, and chunk it
    for file in PDF_FILES:
        print(f"Processing {file}...")
        text = get_pdf_chunks(file)
        chunks = get_pdf_chunks(file, chunk_size=1000, overlap=200)

        # Adding a prefix to each chunk so the LLM knows which file it's from
        chunks_with_metadata = [f"Source: {file}\n\n{c}" for c in chunks]
        all_chunks.extend(chunks_with_metadata)

    print(f"Total chunks created: {len(all_chunks)}")
    
    # Batch embed everything
    embeddings = get_embeddings(all_chunks)
    
    # Save to disk
    np.save("data/embeddings.npy", embeddings)
    with open("data/chunks.pkl", "wb") as f:
        pickle.dump(all_chunks, f)

ingest_all()

Processing data/pdfs/AAPL_FY2024_10-K.pdf...
Processing data/pdfs/AAPL_FY2025_10-K.pdf...
Processing data/pdfs/GOOGL_FY2024_10-K.pdf...
Processing data/pdfs/GOOGL_FY2025_10-K.pdf...
Processing data/pdfs/MSFT_FY2024_10-K.pdf...
Processing data/pdfs/MSFT_FY2025_10-K.pdf...
Total chunks created: 2296
Processing batch 1 (100 chunks)...
Processing batch 2 (100 chunks)...
Processing batch 3 (100 chunks)...
Processing batch 4 (100 chunks)...
Processing batch 5 (100 chunks)...
Processing batch 6 (100 chunks)...
Processing batch 7 (100 chunks)...
Processing batch 8 (100 chunks)...
Processing batch 9 (100 chunks)...
Processing batch 10 (100 chunks)...
Processing batch 11 (100 chunks)...
Processing batch 12 (100 chunks)...
Processing batch 13 (100 chunks)...
Processing batch 14 (100 chunks)...
Processing batch 15 (100 chunks)...
Processing batch 16 (100 chunks)...
Processing batch 17 (100 chunks)...
Processing batch 18 (100 chunks)...
Processing batch 19 (100 chunks)...
Processing batch 20 (100 c

In [31]:
# Define the tools for the Fireworks Model
tools = [
    {
        "type": "function",
        "function": {
            "name": "query_financial_db",
            "description": "Query the SQLite database for structured financial data like revenue, net income, assets, etc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql_query": {"type": "string", "description": "The SQL query to run."}
                },
                "required": ["sql_query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_10k_filings",
            "description": "Search the 10-K text for qualitative info like risks, strategy, and business descriptions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search term for the 10-K filings."}
                },
                "required": ["query"]
            }
        }
    }
]

def agent_chat(user_input):
    messages = [
        {"role": "system", "content": "You are a helpful financial analyst. Use the provided tools to answer questions. If you need both hard numbers and strategic context, call both tools. Always cite your source."},
        {"role": "user", "content": user_input}
    ]
    
    # Call Fireworks with tools
    response = client.chat.completions.create(
        model="accounts/fireworks/models/llama-v3-70b-instruct",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    # Logic to handle tool calls, execute the functions you wrote earlier, 
    # and send the results back to the model for a final summary goes here.
    return response.choices[0].message

In [174]:
def ask_agent(question_text):
    url = "http://localhost:8000/api/chat"
    payload = {"question": question_text}
    response = requests.post(url, json=payload)
    response.raise_for_status() # Check for server errors
    return response.json()

final_questions = [
            "What was Apple's total net revenue in fiscal year 2025?",
            "What were the primary risk factors Apple disclosed related to its supply chain in its fiscal year 2025 10-K filing?",
            "What percentage of Microsoft's total revenue came from the United States in fiscal year 2025?",
            "Which company had the fastest revenue growth rate between their two most recent fiscal years, and what was the growth rate?",
            "How did Apple's Greater China revenue change between fiscal year 2024 and fiscal year 2025, and does the 10-K filing explain why?",
            "Microsoft restructured its segment reporting in August 2024. How are the three segments defined in the fiscal year 2025 10-K, and what was the revenue for each?",
            "According to Alphabet's 10-K filings, what are the key components of 'Google Services' revenue, and how did YouTube advertising revenue change between FY2024 and FY2025?",
            "Which company has the highest current ratio (current assets / current liabilities) in their most recent fiscal year?",
            "Across all three companies, which saw the largest absolute dollar increase in revenue between their two most recent fiscal years? Break down how much of that increase came from each business segment.",
            "Apple's Services segment has been growing as a share of total revenue. Calculate the Services revenue growth rate for each of the last two years, compare it to iPhone revenue growth, and find what Apple's 10-K says about the strategic importance of Services."
            ]
for i, question in enumerate(final_questions):
    print('Answer', i+1)
    print(ask_agent(question)['answer']) 
    print('---\n')

Answer 1
Apple's total net revenue in fiscal year 2025 was $416,161,000,000.
---

Answer 2
The primary risk factors Apple disclosed related to its supply chain in its fiscal year 2025 10-K filing include:

1. Adverse macroeconomic conditions, such as slow growth or recession, high unemployment, inflation, tighter credit, higher interest rates, and currency fluctuations, which can impact consumer spending and the company's ability to obtain components.
2. Supply chain disruptions, including those caused by severe financial problems or other disruptions experienced by the company's sourcing partners or suppliers, which can lead to disruptions or termination of supply and negative impacts on the recoverability of manufacturing process equipment or prepayments.
3. Changes or additions to the supply chain, which require considerable time and resources and involve significant risks and uncertainties, including exposure to additional regulatory and operational risks.
4. The company's reliance

In [128]:
# Check db
db_path = 'data/financials.db'
conn = sqlite3.connect(db_path)

# List all tables in the database
# result = pd.read_sql_query("SELECT * FROM companies LIMIT 10;", conn)
# print(result)

# result = pd.read_sql_query("SELECT distinct fiscal_year, company_ticker FROM income_statements LIMIT 100;", conn)
# print(result)

query = "SELECT companies.name, (income_statements.revenue * 1.0 / LAG(income_statements.revenue) OVER (PARTITION BY companies.name ORDER BY income_statements.fiscal_year) - 1) * 100 AS revenue_growth_rate FROM income_statements JOIN companies ON companies.ticker = income_statements.company_ticker ORDER BY revenue_growth_rate DESC LIMIT 1"
result = pd.read_sql_query(query, conn)
print(result)


conn.close()

                    name  revenue_growth_rate
0  Microsoft Corporation            15.669962
